# RQ1 — Baseline supervised models (regression)

**Research question (RQ1).** How effectively can baseline supervised learning models predict `revenue_million` from movie metadata and production/marketing/ratings features?

**Task:** regression to predict `revenue_million`. **Outputs:** PDF + CSV in `./outputs` (or `/kaggle/working` on Kaggle).

## Methodology (this notebook)

1. Load `global_movies_dataset_1950_2026.csv`.
2. Build a modeling frame: numeric conversion + missing-value handling.
3. Train/test split (75/25, `random_state=42`).
4. Report **MAE**, **RMSE**, **R²**; save tables and figures.

In [1]:
# Setup: paths, load data, modeling frame (Global movies — regression)
from __future__ import annotations

import os
import warnings
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

IS_KAGGLE = os.path.exists("/kaggle/input")
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path(".")
OUT = Path("/kaggle/working") if IS_KAGGLE else Path("outputs")
OUT.mkdir(parents=True, exist_ok=True)
RQ_PREFIX = "RQ01"
RANDOM_STATE = 42
TARGET = "revenue_million"

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 10,
        "figure.titlesize": 14,
    }
)
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)


def find_raw_table_path() -> Path:
    preferred = ("global_movies_dataset_1950_2026.csv",)
    found: list[Path] = []
    if IS_KAGGLE:
        for root, _, files in os.walk(INPUT_ROOT):
            for fn in files:
                p = Path(root) / fn
                if p.suffix.lower() in {".csv"}:
                    found.append(p)
    else:
        for p in INPUT_ROOT.rglob("*"):
            if p.is_file() and p.suffix.lower() in {".csv"}:
                found.append(p)
    for name in preferred:
        for p in found:
            if p.name.lower() == name.lower():
                return p
    for p in found:
        low = p.name.lower()
        if "global" in low and "movies" in low and "1950" in low:
            return p
    if found:
        return found[0]
    raise FileNotFoundError(
        "No CSV found. Add the dataset via Kaggle Add Input or place global_movies_dataset_1950_2026.csv next to this notebook."
    )


def prepare_modeling_df(raw: pd.DataFrame) -> pd.DataFrame:
    d = raw.copy()
    numeric_cols = [
        "release_year",
        "runtime_min",
        "imdb_rating",
        "votes",
        "budget_million",
        "marketing_budget_million",
        "metascore",
        "audience_score",
        "award_nominations",
        "award_wins",
    ]
    for c in numeric_cols:
        if c in d.columns:
            d[c] = pd.to_numeric(d[c], errors="coerce")
    if "franchise_flag" in d.columns:
        d["franchise_flag"] = pd.to_numeric(d["franchise_flag"], errors="coerce")

    cat_cols = [
        "genre",
        "subgenre",
        # high-cardinality fields intentionally excluded for speed
        "country",
        "language",
        "streaming_platform",
    ]
    for c in cat_cols:
        if c in d.columns:
            d[c] = d[c].fillna("missing").astype(str)

    d[TARGET] = pd.to_numeric(d[TARGET], errors="coerce")
    d = d.dropna(subset=[TARGET])
    if len(d) > 30000:
        d = d.sample(30000, random_state=RANDOM_STATE)
    return d


def regression_metrics(y_true, y_pred) -> dict:
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "R2": float(r2_score(y_true, y_pred)),
    }


RAW_PATH = find_raw_table_path()
df = pd.read_csv(RAW_PATH)
MODEL_DF = prepare_modeling_df(df)

# Predictors (exclude obvious leakage/labels)
DEFAULT_FEATURE_COLS = [
    "release_year",
    "runtime_min",
    "imdb_rating",
    "votes",
    "budget_million",
    "marketing_budget_million",
    "metascore",
    "audience_score",
    "award_nominations",
    "award_wins",
    "franchise_flag",
    "genre",
    "subgenre",
    "country",
    "language",
    "streaming_platform",
]
LEAKY_OR_LABEL_COLS = {"roi_pct", "top_100_prob", "blockbuster_flag"}
FEATURE_COLS = [c for c in DEFAULT_FEATURE_COLS if c in MODEL_DF.columns and c not in LEAKY_OR_LABEL_COLS]

print("Loaded:", RAW_PATH)
print("Rows, cols (raw):", df.shape)
print("Rows (modeling, after dropna target):", len(MODEL_DF))
print("Target:", TARGET)
print("#Features:", len(FEATURE_COLS))


Loaded: global_movies_dataset_1950_2026.csv
Rows, cols (raw): (100000, 27)
Rows (modeling, after dropna target): 30000
Target: revenue_million
#Features: 16


In [2]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

X = MODEL_DF[FEATURE_COLS]
y = MODEL_DF[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE
)

from pandas.api.types import is_numeric_dtype

cat_cols = [c for c in FEATURE_COLS if not is_numeric_dtype(MODEL_DF[c])]
num_cols = [c for c in FEATURE_COLS if is_numeric_dtype(MODEL_DF[c])]


def make_preprocessor(scale_numeric: bool = True):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))
    num_pipe = Pipeline(num_steps)
    cat_pipe = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore")),
        ]
    )
    return ColumnTransformer(
        [("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)]
    )


rows = []
for name, model in [
    ("LinearRegression", LinearRegression()),
    (
        "DecisionTreeRegressor",
        DecisionTreeRegressor(
            random_state=RANDOM_STATE, max_depth=14, min_samples_leaf=10
        ),
    ),
    ("KNeighborsRegressor", KNeighborsRegressor(n_neighbors=9, weights="distance")),
]:
    pipe = Pipeline([("prep", make_preprocessor(scale_numeric=True)), ("model", model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({"Model": name, **regression_metrics(y_test, pred)})

tbl = pd.DataFrame(rows)
tbl.to_csv(OUT / f"{RQ_PREFIX}_table_baseline_performance.csv", index=False)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
x = np.arange(len(tbl))
w = 0.25
ax.bar(x - w, tbl["MAE"], width=w, label="MAE (lower better)", color="steelblue")
ax.bar(x, tbl["RMSE"], width=w, label="RMSE (lower better)", color="coral")
ax.bar(x + w, tbl["R2"], width=w, label="R² (higher better)", color="seagreen")
ax.set_xticks(x)
ax.set_xticklabels(tbl["Model"], rotation=15, ha="right")
ax.set_ylabel("Metric value")
ax.set_title("RQ1 — Baseline models: MAE, RMSE, R² (test set)")
ax.legend()
plt.tight_layout()
fig.savefig(OUT / f"{RQ_PREFIX}_fig_baseline_comparison.pdf")
plt.close()

tbl


,Model,MAE,RMSE,R2
0,LinearRegression,92.017129,164.167531,0.343426
1,DecisionTreeRegressor,91.210318,174.509185,0.258100
2,KNeighborsRegressor,91.058404,168.938203,0.304712
